# PlankEye V7.1 — Corner Focus

Notebook Kaggle complet pour entraîner **PlankEye V7.1**, la variante orientée précision des 4 coins.

Ce notebook garde la V7 séparée :

- V7 : `/kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7`
- V7.1 : `/kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7_1`

Modes automatiques :

1. **RESUME V7.1** si un `last.pt` V7.1 existe/restauré.
2. **INIT FROM V7** si aucun V7.1 n'existe mais un checkpoint V7 est disponible.
3. **FRESH V7.1** sinon, depuis ResNet50 ImageNet.

Pour passer V7 → V7.1, on utilise `--init-from` et **pas** `--resume` : les poids sont repris, mais optimizer/scheduler/historique repartent proprement.

In [ ]:
print("=" * 88)
print("PLANKEYE V7.1 - CORNER FOCUS")
print("=" * 88)
print("Notebook initialisé.")
print("La V7 originale ne sera pas écrasée.")
print("=" * 88)

## Cellule 1 — Imports, chemins et configuration

**À vérifier avant lancement :**

- `GITHUB_REPO`
- le chemin du dataset
- `V7_INIT_FILENAME` si tu veux initialiser V7.1 avec un checkpoint V7 particulier
- le dataset de backup V7.1 si tu en crées un

Par défaut, le backup Kaggle V7.1 est désactivé pour éviter d'écraser ton dataset de checkpoints V7.

In [ ]:
from pathlib import Path
import json
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import time

import torch


# ============================================================
# PROJET
# ============================================================

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
REPO = Path("/kaggle/working/Train_Kaggle_plank_Detector")

V7_RUN_DIR = PROJECT / "runs" / "plankeye_v7"
RUN_DIR = PROJECT / "runs" / "plankeye_v7_1"

PROJECT.mkdir(parents=True, exist_ok=True)
V7_RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# GITHUB
# ============================================================

GITHUB_REPO = (
    "https://github.com/"
    "maaxxe/Train_Kaggle_plank_Detector.git"
)


# ============================================================
# DATASET
# ============================================================

DATA_DIR = Path(
    "/kaggle/input/datasets/max778/plankeye/"
    "data_kaggle_2_propre/data_kaggle_2_propre"
)


# ============================================================
# CHECKPOINT V7 UTILISÉ POUR INITIALISER V7.1
# ============================================================

KAGGLE_V7_CHECKPOINT_DATASET = "max778/chekpoints-backbone50"

V7_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/max778/chekpoints-backbone50"),
    Path("/kaggle/input/chekpoints-backbone50"),
]

# Mettre par exemple "epoch_040.pt" plus tard si ce checkpoint
# est meilleur que best.pt pour la géométrie.
V7_INIT_FILENAME = "best.pt"


# ============================================================
# CHECKPOINTS V7.1
# ============================================================

# Crée plus tard un dataset séparé, par exemple :
# max778/checkpoints-plankeye-v71
KAGGLE_V71_CHECKPOINT_DATASET = "max778/checkpoints-plankeye-v71"

V71_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/max778/checkpoints-plankeye-v71"),
    Path("/kaggle/input/checkpoints-plankeye-v71"),
]

# Laisser False tant que le dataset V7.1 n'existe pas.
ENABLE_KAGGLE_BACKUP = False


# ============================================================
# FICHIERS LOCAUX
# ============================================================

LOCAL_MODEL = PROJECT / "model_v7_1.py"
LOCAL_TRAIN = PROJECT / "train_v7_1.py"
LOCAL_SYNC = PROJECT / "checkpoint_sync.py"


# ============================================================
# TRAIN
# ============================================================

NUM_GPUS = 2

EPOCHS = 180
BATCH_SIZE = 1
GRAD_ACCUM = 8
WORKERS = 2
IMAGE_SIZE = 512
VAL_RATIO = 0.10

# Fresh V7.1 depuis ImageNet.
FRESH_LR = 2e-4
FRESH_WARMUP_EPOCHS = 3.0

# Fine-tuning V7 -> V7.1.
FINETUNE_LR = 1e-4
FINETUNE_WARMUP_EPOCHS = 1.0

BACKBONE_LR_MULT = 0.25
WEIGHT_DECAY = 1e-4

UNFREEZE_LAYER4_EPOCH = 4
UNFREEZE_LAYER3_EPOCH = 9
UNFREEZE_ALL_EPOCH = 16

MIN_LR_RATIO = 0.03
MAX_GRAD_NORM = 10.0

SAVE_EVERY = 5
KAGGLE_UPLOAD_EVERY = 5


# ============================================================
# AFFICHAGE
# ============================================================

print("=" * 88)
print("CONFIGURATION V7.1")
print("=" * 88)
print("PROJECT             :", PROJECT)
print("RUN V7              :", V7_RUN_DIR)
print("RUN V7.1            :", RUN_DIR)
print("DATA                :", DATA_DIR)
print("V7 init filename    :", V7_INIT_FILENAME)
print("Backup V7.1 activé :", ENABLE_KAGGLE_BACKUP)
print("GPU demandés        :", NUM_GPUS)
print("Batch / GPU         :", BATCH_SIZE)
print("Grad accum          :", GRAD_ACCUM)
print("Batch effectif      :", BATCH_SIZE * NUM_GPUS * GRAD_ACCUM)
print("=" * 88)

## Cellule 2 — Vérification GPU

In [ ]:
print("=" * 88)
print("GPU")
print("=" * 88)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPU     :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"{props.total_memory / 1024**3:.2f} GiB"
    )

if torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(
        f"❌ {NUM_GPUS} GPU requis, seulement "
        f"{torch.cuda.device_count()} détecté(s). "
        "Active GPU T4 x2 dans Kaggle."
    )

print("✅ Configuration GPU valide.")

## Cellule 3 — Clone / mise à jour du GitHub

In [ ]:
if REPO.exists() and (REPO / ".git").exists():
    print("Repository déjà présent -> git pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only"],
        check=True,
    )
else:
    if REPO.exists():
        shutil.rmtree(REPO)

    print("Clone du repository...")
    subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO)],
        check=True,
    )

print("✅ Git prêt :", REPO)

## Cellule 4 — Copier V7.1 dans le projet Kaggle

Le repository doit contenir :

```text
Plankeye/model/model_v7_1.py
Plankeye/train/train_v7_1.py
Plankeye/train/checkpoint_sync.py
```

In [ ]:
SOURCE_MODEL = REPO / "Plankeye" / "model" / "model_v7_1.py"
SOURCE_TRAIN = REPO / "Plankeye" / "train" / "train_v7_1.py"
SOURCE_SYNC = REPO / "Plankeye" / "train" / "checkpoint_sync.py"

required = {
    "model_v7_1.py": SOURCE_MODEL,
    "train_v7_1.py": SOURCE_TRAIN,
}

print("=" * 88)
print("COPIE V7.1")
print("=" * 88)

for name, source in required.items():
    if not source.exists():
        raise FileNotFoundError(
            f"❌ {name} absent du Git :\n{source}\n\n"
            "Ajoute les fichiers V7.1 dans le repository puis relance cette cellule."
        )

shutil.copy2(SOURCE_MODEL, LOCAL_MODEL)
shutil.copy2(SOURCE_TRAIN, LOCAL_TRAIN)

if SOURCE_SYNC.exists():
    shutil.copy2(SOURCE_SYNC, LOCAL_SYNC)
    print(f"✅ checkpoint_sync.py -> {LOCAL_SYNC}")
else:
    print("⚠️ checkpoint_sync.py absent du Git.")
    if ENABLE_KAGGLE_BACKUP:
        raise FileNotFoundError(SOURCE_SYNC)

print(f"✅ model_v7_1.py     -> {LOCAL_MODEL}")
print(f"✅ train_v7_1.py     -> {LOCAL_TRAIN}")
print("✅ V7.1 prêt.")

## Cellule 5 — Vérification syntaxe et version

In [ ]:
import py_compile
import importlib.util

for path in (LOCAL_MODEL, LOCAL_TRAIN):
    py_compile.compile(str(path), doraise=True)
    print(f"✅ Syntaxe OK : {path.name}")

# Lire MODEL_VERSION sans importer tout le modèle deux fois.
text = LOCAL_MODEL.read_text(encoding="utf-8")
match = re.search(r'MODEL_VERSION\s*=\s*["\']([^"\']+)["\']', text)

if not match:
    raise RuntimeError("MODEL_VERSION introuvable dans model_v7_1.py")

MODEL_VERSION_FILE = match.group(1)

print()
print("MODEL_VERSION :", MODEL_VERSION_FILE)

if not MODEL_VERSION_FILE.startswith("v7.1"):
    raise RuntimeError(
        f"❌ Mauvais modèle copié : {MODEL_VERSION_FILE}"
    )

print("✅ Version V7.1 confirmée.")

## Cellule 6 — Vérifier le dataset

In [ ]:
print("=" * 88)
print("DATASET")
print("=" * 88)

IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

if not DATA_DIR.exists():
    raise FileNotFoundError(DATA_DIR)
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(IMAGES_DIR)
if not LABELS_DIR.is_dir():
    raise FileNotFoundError(LABELS_DIR)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
images = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_exts
)
labels = sorted(LABELS_DIR.glob("*.txt"))

print("Images :", len(images))
print("Labels :", len(labels))
print("Images dir :", IMAGES_DIR)
print("Labels dir :", LABELS_DIR)

if len(images) == 0:
    raise RuntimeError("❌ Aucune image.")
if len(labels) == 0:
    raise RuntimeError("❌ Aucun label.")

print("✅ Dataset accessible.")

## Cellule 7 — Restaurer un checkpoint V7.1 si disponible

Cette cellule ne touche qu'au dossier `runs/plankeye_v7_1`.

Si le dataset V7.1 n'est pas attaché à ce notebook, elle passe simplement à l'étape suivante.

In [ ]:
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


V71_INPUT = first_existing(V71_INPUT_CANDIDATES)
local_v71_last = RUN_DIR / "last.pt"

print("=" * 88)
print("RESTAURATION V7.1")
print("=" * 88)

if local_v71_last.exists():
    print("✅ last.pt V7.1 déjà présent localement.")
    print(f"Taille : {local_v71_last.stat().st_size / 1024**2:.2f} MB")

elif V71_INPUT is not None:
    print("Dataset V7.1 trouvé :", V71_INPUT)

    names = [
        "last.pt",
        "best.pt",
        "best_corners.pt",
        "history.json",
        "history.csv",
        "loss_curve.png",
        "dataset_split.json",
        "run_config.json",
        "model_v7_1.py",
        "train_v7_1.py",
    ]

    restored = 0

    for name in names:
        source = V71_INPUT / name
        destination = RUN_DIR / name

        if source.exists():
            shutil.copy2(source, destination)
            restored += 1
            print(
                f"✅ {name:<24} "
                f"{destination.stat().st_size / 1024**2:>8.2f} MB"
            )

    print()
    print("Fichiers restaurés :", restored)

    if (RUN_DIR / "last.pt").exists():
        print("✅ Reprise V7.1 disponible.")
    else:
        print("⚠️ Dataset V7.1 présent mais last.pt absent.")

else:
    print("ℹ️ Aucun dataset checkpoint V7.1 attaché.")
    print("➡️ Si aucun checkpoint V7.1 local n'existe, on cherchera un V7 pour initialiser V7.1.")

## Cellule 8 — Restaurer le checkpoint V7 de départ

Cette étape est utilisée **uniquement si V7.1 n'a pas encore commencé**.

`V7_INIT_FILENAME` peut être :

- `best.pt`
- `last.pt`
- `epoch_040.pt`
- etc.

In [ ]:
V7_INPUT = first_existing(V7_INPUT_CANDIDATES)

requested_v7 = V7_RUN_DIR / V7_INIT_FILENAME

print("=" * 88)
print("CHECKPOINT V7 POUR INITIALISATION V7.1")
print("=" * 88)

if requested_v7.exists():
    print("✅ Checkpoint V7 déjà local :", requested_v7)

elif V7_INPUT is not None:
    print("Dataset V7 trouvé :", V7_INPUT)

    source = V7_INPUT / V7_INIT_FILENAME

    if not source.exists():
        print(
            f"⚠️ {V7_INIT_FILENAME} absent du dataset. "
            "Recherche de fallback..."
        )

        fallbacks = [
            V7_INPUT / "best.pt",
            V7_INPUT / "last.pt",
        ]

        epoch_files = []
        for p in V7_INPUT.glob("epoch_*.pt"):
            m = re.fullmatch(r"epoch_(\d+)\.pt", p.name)
            if m:
                epoch_files.append((int(m.group(1)), p))

        epoch_files.sort(reverse=True)

        source = next(
            (p for p in fallbacks if p.exists()),
            epoch_files[0][1] if epoch_files else None,
        )

    if source is not None and source.exists():
        destination = V7_RUN_DIR / source.name
        shutil.copy2(source, destination)

        requested_v7 = destination

        print(
            f"✅ V7 restauré : {destination.name} | "
            f"{destination.stat().st_size / 1024**2:.2f} MB"
        )
    else:
        print("⚠️ Aucun checkpoint V7 trouvé dans le dataset.")

else:
    print("ℹ️ Dataset checkpoint V7 non attaché.")


# Si le fichier demandé n'existe pas mais qu'un fallback a été copié,
# la variable requested_v7 pointe déjà vers lui.
V7_INIT_CHECKPOINT = requested_v7 if requested_v7.exists() else None

print()
print("V7_INIT_CHECKPOINT :", V7_INIT_CHECKPOINT)

## Cellule 9 — Déterminer automatiquement le mode de training

In [ ]:
def find_resume_checkpoint(run_dir: Path):
    last_pt = run_dir / "last.pt"
    if last_pt.exists():
        return last_pt

    epochs = []
    for path in run_dir.glob("epoch_*.pt"):
        m = re.fullmatch(r"epoch_(\d+)\.pt", path.name)
        if m:
            epochs.append((int(m.group(1)), path))

    if epochs:
        epochs.sort(key=lambda x: x[0], reverse=True)
        return epochs[0][1]

    best_corners = run_dir / "best_corners.pt"
    if best_corners.exists():
        return best_corners

    best_pt = run_dir / "best.pt"
    if best_pt.exists():
        return best_pt

    return None


RESUME_CHECKPOINT = find_resume_checkpoint(RUN_DIR)

if RESUME_CHECKPOINT is not None:
    TRAIN_MODE = "resume_v71"
elif V7_INIT_CHECKPOINT is not None and V7_INIT_CHECKPOINT.exists():
    TRAIN_MODE = "init_from_v7"
else:
    TRAIN_MODE = "fresh_v71"


print("=" * 88)
print("MODE V7.1")
print("=" * 88)
print("Mode :", TRAIN_MODE)

if TRAIN_MODE == "resume_v71":
    print("Checkpoint :", RESUME_CHECKPOINT)

elif TRAIN_MODE == "init_from_v7":
    print("Checkpoint V7 :", V7_INIT_CHECKPOINT)
    print("➡️ Poids/EMA chargés, mais nouvel optimizer/scheduler/historique.")
    print("➡️ Backbone complet entraînable dès l'epoch 1 de V7.1.")

else:
    print("➡️ Aucun checkpoint disponible.")
    print("➡️ V7.1 démarrera depuis ResNet50 ImageNet.")
    print("➡️ Progressive unfreeze activé.")

## Cellule 10 — Examiner le checkpoint sélectionné

In [ ]:
CHECKPOINT_TO_INSPECT = (
    RESUME_CHECKPOINT
    if TRAIN_MODE == "resume_v71"
    else V7_INIT_CHECKPOINT
)

if CHECKPOINT_TO_INSPECT is None:
    print("ℹ️ Aucun checkpoint à inspecter.")
else:
    print("=" * 88)
    print("CHECKPOINT")
    print("=" * 88)
    print("Fichier :", CHECKPOINT_TO_INSPECT)

    ckpt = torch.load(
        CHECKPOINT_TO_INSPECT,
        map_location="cpu",
        weights_only=False,
    )

    fields = [
        "model_version",
        "epoch_human",
        "epoch",
        "global_optimizer_step",
        "best_val_loss",
        "best_corner_loss",
        "train_loss",
        "val_loss",
        "backbone_stage",
        "timestamp",
    ]

    for field in fields:
        if field in ckpt:
            print(f"{field:<24}: {ckpt.get(field)}")

    print("EMA présent            :", "ema_state_dict" in ckpt)
    print("Optimizer présent      :", "optimizer_state_dict" in ckpt)

    del ckpt

## Cellule 11 — Construire la commande `torchrun`

Réglages :

- **resume V7.1** : reprise complète de l'état d'entraînement ;
- **V7 → V7.1** : LR `1e-4`, warmup 1 epoch, backbone complet dégelé ;
- **fresh V7.1** : LR `2e-4`, warmup 3 epochs, dégel progressif.

In [ ]:
# ============================================================
# CHOIX LR / WARMUP
# ============================================================

if TRAIN_MODE == "fresh_v71":
    LEARNING_RATE = FRESH_LR
    WARMUP_EPOCHS = FRESH_WARMUP_EPOCHS

elif TRAIN_MODE == "init_from_v7":
    LEARNING_RATE = FINETUNE_LR
    WARMUP_EPOCHS = FINETUNE_WARMUP_EPOCHS

else:
    # Les états optimizer/scheduler seront restaurés par --resume.
    # Ces valeurs servent surtout à conserver une config cohérente.
    run_config = RUN_DIR / "run_config.json"
    previous_no_progressive = None

    if run_config.exists():
        try:
            cfg = json.loads(run_config.read_text(encoding="utf-8"))
            previous_no_progressive = bool(
                cfg.get("args", {}).get("no_progressive_unfreeze", False)
            )
        except Exception:
            previous_no_progressive = None

    if previous_no_progressive:
        LEARNING_RATE = FINETUNE_LR
        WARMUP_EPOCHS = FINETUNE_WARMUP_EPOCHS
    else:
        LEARNING_RATE = FRESH_LR
        WARMUP_EPOCHS = FRESH_WARMUP_EPOCHS


# ============================================================
# COMMANDE
# ============================================================

train_command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node",
    str(NUM_GPUS),

    str(LOCAL_TRAIN),

    "--data",
    str(DATA_DIR),

    "--output",
    str(RUN_DIR),

    "--epochs",
    str(EPOCHS),

    "--batch-size",
    str(BATCH_SIZE),

    "--grad-accum",
    str(GRAD_ACCUM),

    "--workers",
    str(WORKERS),

    "--image-size",
    str(IMAGE_SIZE),

    "--val-ratio",
    str(VAL_RATIO),

    "--lr",
    str(LEARNING_RATE),

    "--backbone-lr-mult",
    str(BACKBONE_LR_MULT),

    "--weight-decay",
    str(WEIGHT_DECAY),

    "--unfreeze-layer4-epoch",
    str(UNFREEZE_LAYER4_EPOCH),

    "--unfreeze-layer3-epoch",
    str(UNFREEZE_LAYER3_EPOCH),

    "--unfreeze-all-epoch",
    str(UNFREEZE_ALL_EPOCH),

    "--warmup-epochs",
    str(WARMUP_EPOCHS),

    "--min-lr-ratio",
    str(MIN_LR_RATIO),

    "--max-grad-norm",
    str(MAX_GRAD_NORM),

    "--save-every",
    str(SAVE_EVERY),
]


# ============================================================
# MODE
# ============================================================

if TRAIN_MODE == "resume_v71":
    train_command.extend(
        ["--resume", str(RESUME_CHECKPOINT)]
    )

    run_config = RUN_DIR / "run_config.json"
    if run_config.exists():
        try:
            cfg = json.loads(run_config.read_text(encoding="utf-8"))
            if bool(cfg.get("args", {}).get("no_progressive_unfreeze", False)):
                train_command.append("--no-progressive-unfreeze")
        except Exception:
            pass

elif TRAIN_MODE == "init_from_v7":
    train_command.extend(
        ["--init-from", str(V7_INIT_CHECKPOINT)]
    )
    # Le V7 a déjà appris le backbone.
    train_command.append("--no-progressive-unfreeze")

# fresh_v71 : pas de --resume / --init-from / --no-pretrained


# ============================================================
# BACKUP KAGGLE V7.1
# ============================================================

if ENABLE_KAGGLE_BACKUP:
    if not KAGGLE_V71_CHECKPOINT_DATASET.strip():
        raise RuntimeError(
            "ENABLE_KAGGLE_BACKUP=True mais KAGGLE_V71_CHECKPOINT_DATASET est vide."
        )

    if KAGGLE_V71_CHECKPOINT_DATASET == KAGGLE_V7_CHECKPOINT_DATASET:
        raise RuntimeError(
            "❌ Refus de sauvegarder V7.1 dans le dataset checkpoint V7. "
            "Utilise un dataset V7.1 séparé."
        )

    if not LOCAL_SYNC.exists():
        raise FileNotFoundError(LOCAL_SYNC)

    train_command.extend(
        [
            "--kaggle-dataset",
            KAGGLE_V71_CHECKPOINT_DATASET,
            "--kaggle-upload-every",
            str(KAGGLE_UPLOAD_EVERY),
        ]
    )


command_text = " ".join(str(x) for x in train_command)

print("=" * 88)
print("TRAIN COMMAND V7.1")
print("=" * 88)
print("Mode               :", TRAIN_MODE)
print("LR                 :", LEARNING_RATE)
print("Warmup             :", WARMUP_EPOCHS)
print("GPU                :", NUM_GPUS)
print("Batch / GPU        :", BATCH_SIZE)
print("Grad accum         :", GRAD_ACCUM)
print("Batch effectif     :", BATCH_SIZE * NUM_GPUS * GRAD_ACCUM)
print("Backup Kaggle      :", ENABLE_KAGGLE_BACKUP)
print()
print(command_text)
print("=" * 88)

## Cellule 12 — Vérification finale avant entraînement

In [ ]:
print("=" * 88)
print("PRE-FLIGHT CHECK V7.1")
print("=" * 88)

checks = {
    "model_v7_1.py": LOCAL_MODEL.exists(),
    "train_v7_1.py": LOCAL_TRAIN.exists(),
    "dataset": DATA_DIR.exists(),
    "images": (DATA_DIR / "images").exists(),
    "labels": (DATA_DIR / "labels").exists(),
    f"GPU x{NUM_GPUS}": torch.cuda.device_count() >= NUM_GPUS,
}

if ENABLE_KAGGLE_BACKUP:
    checks["checkpoint_sync.py"] = LOCAL_SYNC.exists()

all_ok = True

for name, state in checks.items():
    print(("✅" if state else "❌"), name)
    all_ok &= bool(state)

if "train_v7_1.py" not in command_text:
    print("❌ La commande ne lance pas train_v7_1.py")
    all_ok = False

if "train_v7.py " in command_text or command_text.endswith("train_v7.py"):
    print("❌ Ancien train_v7.py détecté")
    all_ok = False

if TRAIN_MODE == "resume_v71":
    if "--resume" not in train_command:
        print("❌ --resume absent")
        all_ok = False
    if "--init-from" in train_command:
        print("❌ --init-from ne doit pas être présent pendant un resume V7.1")
        all_ok = False

if TRAIN_MODE == "init_from_v7":
    if "--init-from" not in train_command:
        print("❌ --init-from absent")
        all_ok = False
    if "--resume" in train_command:
        print("❌ --resume ne doit pas être utilisé pour V7 -> V7.1")
        all_ok = False
    if "--no-progressive-unfreeze" not in train_command:
        print("❌ Le backbone devrait être complètement entraînable.")
        all_ok = False

if TRAIN_MODE == "fresh_v71" and "--no-pretrained" in train_command:
    print("❌ --no-pretrained interdit en fresh.")
    all_ok = False

print()

if not all_ok:
    raise RuntimeError("❌ Pre-flight V7.1 échoué.")

print("✅ Tout est prêt pour lancer PlankEye V7.1 Corner Focus.")

## Cellule 13 — LANCEMENT V7.1

Le bouton **Stop** de Kaggle peut interrompre proprement `torchrun`.

Les checkpoints sont écrits dans :

```text
/kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7_1
```

Le fichier important pour la précision géométrique est notamment :

```text
best_corners.pt
```

In [ ]:
# ============================================================
# ENVIRONNEMENT DDP
# ============================================================

train_env = os.environ.copy()
train_env["OMP_NUM_THREADS"] = "2"
train_env["PYTHONUNBUFFERED"] = "1"
train_env["NCCL_SOCKET_IFNAME"] = "lo"
train_env["NCCL_SOCKET_FAMILY"] = "AF_INET"


print("=" * 88)
print("PLANKEYE V7.1 - TRAINING CORNER FOCUS")
print("=" * 88)
print("Mode :", TRAIN_MODE)
print()
print("Commande :")
print(command_text)
print()
print("Pour arrêter : bouton Stop de Kaggle.")
print("=" * 88)


start_time = time.time()
interrupted = False
return_code = None

process = subprocess.Popen(
    train_command,
    cwd=PROJECT,
    env=train_env,
    start_new_session=True,
)

try:
    return_code = process.wait()

except KeyboardInterrupt:
    interrupted = True

    print()
    print("=" * 88)
    print("🛑 ARRÊT MANUEL")
    print("=" * 88)

    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

    try:
        return_code = process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        print("⚠️ Arrêt forcé...")
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        return_code = process.wait()


duration = time.time() - start_time

print()
print("=" * 88)
print("FIN DU PROCESSUS")
print("=" * 88)
print(f"Durée       : {duration / 60:.2f} min")
print(f"Code retour : {return_code}")
print(f"Interrompu  : {interrupted}")
print("=" * 88)


# ============================================================
# BACKUP MANUEL APRÈS INTERRUPTION, UNIQUEMENT SI CONFIGURÉ
# ============================================================

if (
    interrupted
    and ENABLE_KAGGLE_BACKUP
    and LOCAL_SYNC.exists()
    and (RUN_DIR / "last.pt").exists()
):
    print()
    print("☁️ Backup du dernier checkpoint terminé...")

    backup_command = [
        sys.executable,
        str(LOCAL_SYNC),
        "--run-dir",
        str(RUN_DIR),
        "--dataset",
        KAGGLE_V71_CHECKPOINT_DATASET,
        "--message",
        "PlankEye V7.1 Corner Focus - sauvegarde après arrêt manuel",
    ]

    result = subprocess.run(
        backup_command,
        cwd=PROJECT,
        env=train_env,
    )

    print("Code backup :", result.returncode)

## Cellule 14 — Voir les fichiers sauvegardés

In [ ]:
print("=" * 88)
print("FICHIERS V7.1")
print("=" * 88)

if not RUN_DIR.exists():
    print("Aucun dossier de run.")
else:
    files = sorted(
        p for p in RUN_DIR.iterdir()
        if p.is_file()
    )

    for path in files:
        print(
            f"{path.name:<42} "
            f"{path.stat().st_size / 1024**2:>10.2f} MB"
        )

## Cellule 15 — Examiner `last.pt`, `best.pt` et `best_corners.pt`

In [ ]:
for checkpoint_name in (
    "last.pt",
    "best.pt",
    "best_corners.pt",
):
    path = RUN_DIR / checkpoint_name

    print()
    print("=" * 88)
    print(checkpoint_name)
    print("=" * 88)

    if not path.exists():
        print("Absent.")
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    fields = [
        "model_version",
        "epoch_human",
        "global_optimizer_step",
        "best_val_loss",
        "best_corner_loss",
        "train_loss",
        "val_loss",
        "backbone_stage",
        "timestamp",
    ]

    for field in fields:
        if field in ckpt:
            print(f"{field:<24}: {ckpt.get(field)}")

    print("EMA :", "ema_state_dict" in ckpt)

    del ckpt

## Cellule 16 — Résumé de l'historique

In [ ]:
import pandas as pd

history_csv = RUN_DIR / "history.csv"

if not history_csv.exists():
    print("ℹ️ history.csv absent.")
else:
    df = pd.read_csv(history_csv)

    print("Epochs enregistrés :", len(df))
    print("Colonnes disponibles :")
    print(list(df.columns))

    preferred = [
        "epoch",
        "train_total",
        "val_total",
        "train_corner_delta",
        "val_corner_delta",
        "train_corner_abs",
        "val_corner_abs",
        "train_reconstruction",
        "val_reconstruction",
        "train_geometry",
        "val_geometry",
        "train_corner_focus",
        "val_corner_focus",
    ]

    cols = [c for c in preferred if c in df.columns]

    print()
    if cols:
        display(df[cols].tail(15))
    else:
        display(df.tail(15))

## Cellule 17 — Courbes principales V7.1

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history_csv = RUN_DIR / "history.csv"

if not history_csv.exists():
    print("ℹ️ history.csv absent.")
else:
    df = pd.read_csv(history_csv)

    epoch_col = "epoch" if "epoch" in df.columns else None

    if epoch_col is None:
        x = range(1, len(df) + 1)
    else:
        x = df[epoch_col]

    pairs = [
        ("train_total", "val_total", "Total loss"),
        ("train_corner_focus", "val_corner_focus", "Corner focus"),
        ("train_reconstruction", "val_reconstruction", "Reconstruction"),
        ("train_corner_delta", "val_corner_delta", "Corner delta"),
        ("train_corner_abs", "val_corner_abs", "Corner abs"),
        ("train_geometry", "val_geometry", "Geometry"),
    ]

    plotted = 0

    for train_col, val_col, title in pairs:
        available = [
            c for c in (train_col, val_col)
            if c in df.columns
        ]

        if not available:
            continue

        plt.figure(figsize=(10, 5))

        if train_col in df.columns:
            plt.plot(x, df[train_col], label=train_col)

        if val_col in df.columns:
            plt.plot(x, df[val_col], label=val_col)

        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

        plotted += 1

    if plotted == 0:
        print("Aucune colonne de loss reconnue.")

## Cellule 18 — Backup manuel V7.1 sur Kaggle

N'exécute cette cellule que si :

1. tu as créé un dataset Kaggle **séparé** pour V7.1 ;
2. `ENABLE_KAGGLE_BACKUP = True` ;
3. `KAGGLE_V71_CHECKPOINT_DATASET` contient le bon slug.

Cette cellule refuse d'utiliser le dataset checkpoint V7.

In [ ]:
if not ENABLE_KAGGLE_BACKUP:
    print(
        "ℹ️ Backup V7.1 désactivé. "
        "Mets ENABLE_KAGGLE_BACKUP=True dans la Cellule 1 "
        "après avoir créé le dataset dédié."
    )

else:
    if KAGGLE_V71_CHECKPOINT_DATASET == KAGGLE_V7_CHECKPOINT_DATASET:
        raise RuntimeError(
            "❌ Le dataset V7.1 doit être différent du dataset V7."
        )

    if not LOCAL_SYNC.exists():
        raise FileNotFoundError(LOCAL_SYNC)

    if not (RUN_DIR / "last.pt").exists():
        raise FileNotFoundError(RUN_DIR / "last.pt")

    backup_command = [
        sys.executable,
        str(LOCAL_SYNC),
        "--run-dir",
        str(RUN_DIR),
        "--dataset",
        KAGGLE_V71_CHECKPOINT_DATASET,
        "--message",
        "PlankEye V7.1 Corner Focus - checkpoint manuel",
    ]

    print(" ".join(str(x) for x in backup_command))

    result = subprocess.run(
        backup_command,
        cwd=PROJECT,
        env=os.environ.copy(),
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Backup Kaggle échoué : code {result.returncode}"
        )

    print("✅ Backup V7.1 terminé.")

# Utilisation recommandée

Pour ton cas actuel :

1. laisse d'abord la V7 actuelle aller jusqu'à environ **35–40 epochs** ;
2. teste plusieurs checkpoints V7 avec ton évaluateur ;
3. mets le meilleur nom dans `V7_INIT_FILENAME`, par exemple `epoch_040.pt` ;
4. démarre ce notebook V7.1 ;
5. surveille particulièrement `best_corners.pt`, `val_corner_focus`, `val_reconstruction` et `val_geometry`.

La V7.1 est conçue pour renforcer la précision des quatre coins sans écraser le run V7.